## 实现一个最简单的4柱八字计算器

!pip install lunar_python

### API调用

In [7]:
from lunar_python.util import LunarUtil
zhis = ["子","丑","寅","卯","辰","巳","午","未","申","酉","戌","亥"]
for zhi in zhis:
    print(f'{zhi}:{LunarUtil.ZHI_HIDE_GAN[zhi]}')


子:['癸']
丑:['己', '癸', '辛']
寅:['甲', '丙', '戊']
卯:['乙']
辰:['戊', '乙', '癸']
巳:['丙', '庚', '戊']
午:['丁', '己']
未:['己', '丁', '乙']
申:['庚', '壬', '戊']
酉:['辛']
戌:['戊', '辛', '丁']
亥:['壬', '甲']


### 四柱八字计算器

In [1]:
from ipywidgets import HBox, VBox, Text, HTML, Button
from IPython.display import display
from lunar_python.util import LunarUtil


# 创建文本框
texts = [[HTML(value='年干神', layout=dict(width='50px', height='30px')),
          Text(value='甲', layout=dict(width='50px', height='20px')),
          Text(value='寅', layout=dict(width='50px', height='20px')),
          HTML(value='年支神', layout=dict(width='50px', height='90px'))],
         [HTML(value='月干神', layout=dict(width='50px', height='30px')),
          Text(value='甲', layout=dict(width='50px', height='20px')),
          Text(value='子', layout=dict(width='50px', height='20px')),
          HTML(value='月支神', layout=dict(width='50px', height='90px'))],
         [HTML(value='日干神', layout=dict(width='50px', height='30px')),
          Text(value='甲', layout=dict(width='50px', height='20px')),
          Text(value='辰', layout=dict(width='50px', height='20px')),
          HTML(value='日支神', layout=dict(width='50px', height='90px'))],
        [HTML(value='时干神', layout=dict(width='50px', height='30px')),
          Text(value='乙', layout=dict(width='50px', height='20px')),
          Text(value='丑', layout=dict(width='50px', height='20px')),
          HTML(value='时支神', layout=dict(width='50px', height='90px'))],
         [HTML(value='备注', layout=dict(width='300px', height='30px')),
          HTML(value='', layout=dict(width='300px', height='20px')),
          HTML(value='', layout=dict(width='300px', height='20px')),
          HTML(value='', layout=dict(width='300px', height='90px'))]]

# 创建垂直布局
vboxes = [VBox(text, layout=dict(width='60px')) for text in texts]
hbox = HBox(vboxes, layout=dict(height='200px'))
vboxes[-1].layout=dict(width='300px')

#print(vboxes[-1])

# 创建按钮
button = Button(description='更新十神')

# 显示布局
display(hbox)
display(button)

# 定义十神计算函数
def get_shishen(daygan, gans):
    shens = []
    for gan in gans:
        _gg = daygan + gan
        _shen = LunarUtil.SHI_SHEN[_gg]
        # 结果包含藏干名称
        #shens.append(gan +': ' + _shen)
        shens.append(_shen)
    return shens

# 获取五行属性
def get_wuxing(zhigan):
    try:
        return LunarUtil.WU_XING_GAN[zhigan]
    except KeyError:
        return LunarUtil.WU_XING_ZHI[zhigan]
    

# 五行颜色
def color_wuxing(wuxing):
    dict = {'金': 'gold', '木': 'green', '水': 'blue', '火': 'red', '土': 'brown'}
    color = dict.get(wuxing)
    return color     

# 获得阴阳
def get_yinyang(ganzhi):
    ## 阴阳计算
    YinYang = {
    '甲':'阳',
    '丙':'阳',
    '戊':'阳',
    '庚':'阳',
    '壬':'阳',
    
    '子':'阳',
    '寅':'阳',
    '辰':'阳',
    '午':'阳',
    '申':'阳',
    '戌':'阳',
    
    '乙':'阴',
    '丁':'阴',
    '己':'阴',
    '辛':'阴',
    '癸':'阴',
    
    '丑':'阴',
    '卯':'阴',
    '巳':'阴',
    '未':'阴',
    '酉':'阴',
    '亥':'阴'
    }
    return YinYang[ganzhi]

def rizhu_yinyang(daygan):
    # 更新日主阴阳
    day_yinyang = get_yinyang(daygan)
    texts[2][0].value = "日主"
    if day_yinyang=="阳":
        texts[2][1].value = daygan + "   +"
    else:
        texts[2][1].value = daygan + "   -"

def jianlu():
    # 建禄，日主与月令之间判断，
    # texts[2][1].value texts[1][2]
    # 以$符号来显示
    daygan = texts[2][1].value
    yueling = texts[1][2].value
    if(LunarUtil.LU[daygan] == yueling):
        texts[1][2].value = yueling + '  $'

def yangren():
    # 羊刃
    # 甲生卯月​，丙戊生午月，庚生酉月，壬生子月
    REN = {
        "甲":"卯",
        "丙":"午",
        "戊":"午",
        "庚":"酉",
        "壬":"子"
    }
    # texts[2][1].value texts[1][2]
    # 以符号 ✘ 来显示
    daygan = texts[2][1].value
    yueling = texts[1][2].value
    if(REN.get(daygan) == yueling):
        texts[1][2].value = yueling + '  ✘'

def dizhi_sanhe(a,b,c):
    # 地支三合
    # 寅午戌三合火局、巳酉丑三合金局、申子辰三合水局、亥卯未三合木
    # 先将地支排序
    DIZHI_OFFSET = {
        "子": 1,
        "丑": 2,
        "寅": 3,
        "卯": 4,
        "辰": 5,
        "巳": 6,
        "午": 7,
        "未": 8,
        "申": 9,
        "酉": 10,
        "戌": 11,
        "亥": 12
    }

    DIZHI_SANHE ={
        "寅午戌": "火",
        "巳酉丑": "金",
        "申子辰": "水",
        "亥卯未": "木"
    }
    # 如果在DIZHISANHE中，返回对应值
    # 否则返回None
    # 将输入的地支转换为数值
    values = sorted([DIZHI_OFFSET.get(a, 0), DIZHI_OFFSET.get(b, 0), DIZHI_OFFSET.get(c, 0)])
    
    # 检查是否形成三合局
    for sanhe in DIZHI_SANHE:
        sanhe_values = sorted([int(DIZHI_OFFSET[char]) for char in sanhe])
        if values == sanhe_values:
            return sanhe + "三合" + DIZHI_SANHE[sanhe]
    return None

def dizhi_sanhe_set():
    a = texts[0][2].value
    b = texts[1][2].value
    c = texts[2][2].value
    d = texts[3][2].value
    if dizhi_sanhe(a,b,c):
        texts[4][2].value = dizhi_sanhe(a,b,c)
    if dizhi_sanhe(a,b,d):
        texts[4][2].value = dizhi_sanhe(a,b,d)
    if dizhi_sanhe(d,b,c):
        texts[4][2].value = dizhi_sanhe(d,b,c)
    texts[4][2].value = '<span style="font-size:10px;">' + texts[4][2].value + '</span>' 

def dizhi_chong(a,b):
    # 地支有六对相冲，分别是“子午、卯酉、辰戌、丑未、寅申、巳亥”
    DIZHI_CHONG = {
        "子":"午",
        "午":"子",
        "卯":"酉",
        "酉":"卯",
        "辰":"戌",
        "戌":"辰",
        "丑":"未",
        "未":"丑",
        "寅":"申",
        "申":"寅",
        "巳":"亥",
        "亥":"巳"
    }
    if DIZHI_CHONG.get(a) == b:
        return f"{a}{b}相冲"
    else:
        return None

#dizhi_chong("丑","子")

def dizhi_chong_set():
    #print("chong called.")
    a = texts[0][2].value
    b = texts[1][2].value
    c = texts[2][2].value
    d = texts[3][2].value
    chong_each = ""

    if dizhi_chong(a,b):
        chong_each += dizhi_chong(a,b)
    if dizhi_chong(a,c):
        chong_each += dizhi_chong(a,c)
    if dizhi_chong(a,d):
        chong_each += dizhi_chong(a,d)
    if dizhi_chong(b,c):
        chong_each += dizhi_chong(b,c)
    if dizhi_chong(b,d):
        chong_each += dizhi_chong(b,d)
    if dizhi_chong(c,d):
        chong_each += dizhi_chong(c,d)
    if chong_each:
        texts[4][2].value += '|'+'<span style="font-size:10px;color:red">' + chong_each + '</span>' 
    
def dizhi_hai(a,b):
    # 地支有六对相害，分别是“子未， 丑午， 寅巳， 卯辰， 申亥， 酉戌”
    DIZHI_HAI = {
        "子":"未",
        "未":"子",
        "丑":"午",
        "午":"丑",
        "寅":"巳",
        "巳":"寅",
        "卯":"辰",
        "辰":"卯",
        "申":"亥",
        "亥":"申",
        "酉":"戌",
        "戌":"酉"
    }
    if DIZHI_HAI.get(a) == b:
        return f"{a}{b}相害"
    else:
        return None
    
def dizhi_hai_set():
    #print("chong called.")
    a = texts[0][2].value
    b = texts[1][2].value
    c = texts[2][2].value
    d = texts[3][2].value
    hai_each = ""

    if dizhi_hai(a,b):
        hai_each += dizhi_hai(a,b)
    if dizhi_hai(a,c):
        hai_each += dizhi_hai(a,c)
    if dizhi_hai(a,d):
        hai_each += dizhi_hai(a,d)
    if dizhi_hai(b,c):
        hai_each += dizhi_hai(b,c)
    if dizhi_hai(b,d):
        hai_each += dizhi_hai(b,d)
    if dizhi_hai(c,d):
        hai_each += dizhi_hai(c,d)
    
    if hai_each:
        texts[4][2].value += '|'+'<span style="font-size:10px;color:red">' + hai_each + '</span>' 
   
def tiangan_huahe(a,b):
    # 天干合化
    # 甲己合化土，乙庚合化金，丙辛合化水，丁壬合化木，戊癸合化火
    TIANGAN_HEHUA ={
        "甲己": "土",
        "己甲": "土",
        "乙庚": "金",
        "庚乙": "金",
        "丙辛": "水",
        "辛丙": "水",
        "丁壬": "木",
        "壬丁": "木",
        "戊癸": "火",
        "癸戊": "火"
    }
    tiangans = a + b
    huahe = TIANGAN_HEHUA.get(tiangans)
    if huahe:
        return tiangans + "合" + huahe
    else:
        return None

def tiangan_huahe_set():
    # 只计算相邻天干合化

    a = texts[0][1].value
    b = texts[1][1].value
    c = texts[2][1].value
    d = texts[3][1].value

    nianyue_he = tiangan_huahe(a,b)
    yueri_he = tiangan_huahe(b,c)
    rishi_he = tiangan_huahe(c,d)
    shinian_he = tiangan_huahe(d,a) # 时干与年干遥合，看作一个圈

    all_he = ''
    if nianyue_he:
        all_he += nianyue_he +"|"
    if yueri_he:
        all_he += yueri_he +"|"
    if rishi_he:
        all_he += rishi_he +"|"
    if shinian_he:
        all_he += shinian_he +"|"
    texts[4][1].value = '<span style="font-size:10px;">' + all_he + '</span>'

def kongwang_set():
    # 显示空亡
    # 位于日支之下
    daygan = texts[2][1].value
    dayzhi = texts[2][2].value
    dayganzhi = daygan[0] + dayzhi
    kongwang = LunarUtil.getXunKong(dayganzhi)
    if kongwang:
        texts[2][3].value += "\n"+"空: " + kongwang

# 定义回调函数
def update_text(i):
    texts[2][1].value = texts[2][1].value[0] #去除日干显示中的额外信息
    texts[1][2].value = texts[1][2].value[0] #去除月令显示中的额外信息

    daygan = texts[2][1].value
    zhi = texts[i][2].value
    gan = texts[i][1].value
    # gan
    texts[i][0].value = get_shishen(daygan, [gan])[0]

    # cang gan
    # 调整地支藏干中气，余气，新表
    ZHI_HIDE_GAN_NEW = {
        "子": ["癸"],
        "丑": ["己", "辛", "癸"],
        "寅": ["甲", "丙", "戊"],
        "卯": ["乙"],
        "辰": ["戊", "癸", "乙"],
        "巳": ["丙", "庚", "戊"],
        "午": ["丁", "己"],
        "未": ["己", "乙","丁"],
        "申": ["庚", "壬", "戊"],
        "酉": ["辛"],
        "戌": ["戊", "丁","辛"],
        "亥": ["壬", "甲"]
    }

    #gans = LunarUtil.ZHI_HIDE_GAN[zhi]
    gans = ZHI_HIDE_GAN_NEW[zhi]
    gans_color = [color_wuxing(get_wuxing(gan)) for gan in gans]
    shens = get_shishen(daygan, gans)

    # 这里增加span标签，使得藏干颜色显示
    ganWithShen = ['<span style="color:'+ gan_color +'">' + gan + '</span>' + ': ' + shen for gan, shen, gan_color in zip(gans, shens,gans_color)]
    #ganWithShen = [gan + ': ' + shen for gan, shen in zip(gans, shens)]
    
    # xianshi canggan
    texts[i][3].value = '<br>'.join(ganWithShen)

    # rizhu
    #texts[2][0].value = "日主"
    # 更新颜色
    for i in range(4):
        for j in range(1,3):
            wuxing = get_wuxing(texts[i][j].value)
            color = color_wuxing(wuxing)
            texts[i][j].style = {'text_color':color}



# 按钮点击事件
def on_button_clicked(b):
    # update all cell
    for i in range(4):
        update_text(i)

    # 清空备注
    texts[4][1].value = ''
    texts[4][2].value = ''

    # 天干合化
    tiangan_huahe_set()
    # 地支三合
    dizhi_sanhe_set()
    # 地支相冲
    dizhi_chong_set()
    # 地支相害
    dizhi_hai_set()
    # update rigan
    daygan = texts[2][1].value

    # 阳刃
    yangren()
    # 建禄
    jianlu()
    # 日主阴阳
    rizhu_yinyang(daygan)
    # 空亡
    kongwang_set()



    

# 监听按钮点击事件
button.on_click(on_button_clicked)


Button(description='更新十神', style=ButtonStyle())